# Fractional Reasoning

Replicates **"Fractional Reasoning via Latent Steering Vectors Improves Inference Time Compute"** ([arXiv:2506.15882](https://arxiv.org/abs/2506.15882)) on Qwen2.5-1.5B-Instruct, end to end in one engine:

1. **Construction** — hidden states are captured on 20 MATH500 problems paired with a slow-thinking vs a fast-thinking instruction, and the latent deeper-reasoning direction is extracted as a PCA control vector (`MATH500.gguf`).
2. **Steering** — reapplying it with a positive scale lengthens the chain of thought, reported as the mean generated length over 100 MATH-500 problems (`math500.json`). The paper also shortens reasoning with small negative scales; that direction is not stable at this 1.5B model scale, so it is not demoed here.

Uses the current EasySteer steering API (`SteeringSpec` / `VectorSpec` / `ApplySpec`).

**Execution note:** Outputs are cleared after the API migration. Run the notebook from top to bottom to obtain results for your environment.


In [ ]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")  # quiet engine boot logs

from vllm import LLM, SamplingParams
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec

MODEL = os.environ.get("EASYSTEER_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")  # Qwen/Qwen2.5-1.5B-Instruct

# One engine serves both construction (capture) and steering.
llm = LLM(
    model=MODEL,
    tensor_parallel_size=int(os.environ.get("EASYSTEER_TP", "1")),
    enable_steer_vector=True,
    steer_algorithms=["direct"],
)

## Vector construction

In [ ]:
import json

with open("math500_problems.json", encoding="utf-8") as f:
    problems = json.load(f)

slow_thinking = "Walk me through your complete reasoning process step by step. Output as many procedures as you can."
fast_thinking = "Please directly answer the question."

# The assistant turn is pre-seeded with "That" so the captured last
# position sits inside the answer, conditioned on the instruction.
template = "<|im_start|>user\n%s\n%s<|im_end|>\n<|im_start|>assistant\nThat"
texts = [template % (p, slow_thinking) for p in problems] + \
        [template % (p, fast_thinking) for p in problems]

In [ ]:
from easysteer.capture import capture
from vllm.capture import SelectSpec

# Only the last prompt row feeds the extractor, so select it at the source.
result = capture(
    llm,
    texts,
    select=SelectSpec(prompt_positions=[-1]),
    steering=False,
)

In [ ]:
from easysteer.extraction import extract

# Allow the exact estimator to process up to 500 positive/negative pairs.
# Retain exact pair-centered PCA, with each positive paired to the
# corresponding negative. Pool selected tokens before fitting PCA.
labels = [True] * len(problems) + [False] * len(problems)
control_vector = extract(
    result,
    labels,
    method="pca",
    variant="center",
    max_working_bytes=512 * 1024**2,
    token_pos=-1,
    normalize=True,
)
control_vector.export_gguf("MATH500.gguf")


## Steering

In [ ]:
# Baseline: no steering. As in the experiment section, evaluate on
# 100 MATH-500 problems and report only the mean generated length —
# report the observed aggregate for this run.
with open("math500.json", encoding="utf-8") as f:
    eval_problems = [x["problem"] for x in json.load(f)][:100]

prompt_template = "<|im_start|>user\n%s<|im_end|>\n<|im_start|>assistant\n"
eval_texts = [prompt_template % p for p in eval_problems]
params = SamplingParams(temperature=0, max_tokens=8192, skip_special_tokens=False)


def mean_tokens(outputs):
    return sum(len(o.outputs[0].token_ids) for o in outputs) / len(outputs)


baseline = mean_tokens(llm.generate(eval_texts, params, use_tqdm=False, steering=False))
print(f"Baseline mean tokens: {baseline:.0f}")

In [ ]:
# Positive scale amplifies the deeper-reasoning direction -> a longer,
# slower chain of thought. normalize=True is the paper's norm
# rescaling: the steered hidden state keeps its original norm.
steering_slow = SteeringSpec(
    vectors=[
        VectorSpec(
            source="MATH500.gguf",
            scale=2.0,
            layers=list(range(28)),
            normalize=True,
            apply=ApplySpec(prompt="all", generation="all"),
        )
    ],
)
slow = mean_tokens(llm.generate(eval_texts, params, steering=steering_slow,
                                use_tqdm=False))
print(f"Slow mean tokens: {slow:.0f} ({(slow / baseline - 1) * 100:+.0f}%)")